In [30]:
layer_name_map = {
    "wte.weight": "wte",
    "wpe.weight": "wpe",
}

block_map = {
    "h.{i}.ln_1.weight": "h.{i}.ln1.weight",
    "h.{i}.ln_1.bias": "h.{i}.ln1.bias",
    "h.{i}.attn.bias": "h.{i}.attn.causal.mask",
    "h.{i}.attn.c_attn.weight": "h.{i}.qkv.weight",
    "h.{i}.attn.c_attn.bias": "h.{i}.qkv.bias",
    "h.{i}.attn.c_proj.weight": "h.{i}.attn.proj.weight",
    "h.{i}.attn.c_proj.bias": "h.{i}.attn.proj.bias",
    "h.{i}.ln_2.weight": "h.{i}.ln2.weight",
    "h.{i}.ln_2.bias": "h.{i}.ln2.bias",
    "h.{i}.mlp.c_fc.weight": "h.{i}.mlp.up.weight",
    "h.{i}.mlp.c_fc.bias": "h.{i}.mlp.up.bias",
    "h.{i}.mlp.c_proj.weight": "h.{i}.mlp.down.weight",
    "h.{i}.mlp.c_proj.bias": "h.{i}.mlp.down.bias",
}

for i in range(12):
    for src, dst in block_map.items():
        layer_name_map[src.format(i=i)] = dst.format(i=i)

layer_name_map["ln_f.weight"] = "ln.weight"
layer_name_map["ln_f.bias"] = "ln.bias"

layer_name_map

{'wte.weight': 'wte',
 'wpe.weight': 'wpe',
 'h.0.ln_1.weight': 'h.0.ln1.weight',
 'h.0.ln_1.bias': 'h.0.ln1.bias',
 'h.0.attn.bias': 'h.0.attn.causal.mask',
 'h.0.attn.c_attn.weight': 'h.0.qkv.weight',
 'h.0.attn.c_attn.bias': 'h.0.qkv.bias',
 'h.0.attn.c_proj.weight': 'h.0.attn.proj.weight',
 'h.0.attn.c_proj.bias': 'h.0.attn.proj.bias',
 'h.0.ln_2.weight': 'h.0.ln2.weight',
 'h.0.ln_2.bias': 'h.0.ln2.bias',
 'h.0.mlp.c_fc.weight': 'h.0.mlp.up.weight',
 'h.0.mlp.c_fc.bias': 'h.0.mlp.up.bias',
 'h.0.mlp.c_proj.weight': 'h.0.mlp.down.weight',
 'h.0.mlp.c_proj.bias': 'h.0.mlp.down.bias',
 'h.1.ln_1.weight': 'h.1.ln1.weight',
 'h.1.ln_1.bias': 'h.1.ln1.bias',
 'h.1.attn.bias': 'h.1.attn.causal.mask',
 'h.1.attn.c_attn.weight': 'h.1.qkv.weight',
 'h.1.attn.c_attn.bias': 'h.1.qkv.bias',
 'h.1.attn.c_proj.weight': 'h.1.attn.proj.weight',
 'h.1.attn.c_proj.bias': 'h.1.attn.proj.bias',
 'h.1.ln_2.weight': 'h.1.ln2.weight',
 'h.1.ln_2.bias': 'h.1.ln2.bias',
 'h.1.mlp.c_fc.weight': 'h.1.mlp.up.

In [33]:
import json
import struct

path = "/Users/uonliaquat/Downloads/gpt2.safetensors"

with open(path, "rb") as f:
    header_size = struct.unpack("<Q", f.read(8))[0]
    header = json.loads(f.read(header_size))

data_start = 8 + header_size

with open("./gpt2.zg", "w") as gpt2_zg:
    for name, info in header.items():
        if name == "__metadata__":
            continue

        name = layer_name_map[name]

        start, end = info["data_offsets"]

        start += data_start
        end += data_start

        nbytes = end - start

        gpt2_zg.write(
            f"{name:<40}{start},{end}\n"
        )

        print(
            f"{name:<50} start={start:<12} end={end:<12} nbytes={nbytes}"
        )

h.3.ln2.bias                                       start=202434515    end=202437587    nbytes=3072
h.10.ln1.weight                                    start=223168467    end=223171539    nbytes=3072
h.2.qkv.weight                                     start=541027283    end=548105171    nbytes=7077888
h.4.attn.proj.weight                               start=18968531     end=21327827     nbytes=2359296
h.7.attn.causal.mask                               start=263353299    end=267547603    nbytes=4194304
h.4.qkv.bias                                       start=150989779    end=150998995    nbytes=9216
h.2.attn.causal.mask                               start=516095955    end=520290259    nbytes=4194304
h.5.mlp.down.bias                                  start=21327827     end=21330899     nbytes=3072
h.1.attn.proj.weight                               start=513730515    end=516089811    nbytes=2359296
h.4.ln1.weight                                     start=206638035    end=206641107    nbytes=

In [35]:
from safetensors import safe_open

path = "/Users/uonliaquat/Downloads/gpt2.safetensors"
tensor_name = "wpe.weight"

with safe_open(path, framework="pt", device="cpu") as f:
    tensor = f.get_tensor(tensor_name)
    flat = tensor.flatten()

    print(tensor_name)

    # First 10
    print(
        ", ".join(f"{x.item():.3f}" for x in flat[:10]) + ","
    )

    # Last 10
    print(
        ", ".join(f"{x.item():.3f}" for x in flat[-10:]) + ","
    )

wpe.weight
-0.019, -0.197, 0.004, 0.011, 0.064, -0.105, 0.037, -0.168, -0.049, -0.056,
-0.008, 0.001, 0.004, -0.002, -0.000, -0.003, 0.002, -0.005, -0.002, -0.006,
